# Model Training

This notebook trains the baseline fraud-detection models that will be evaluated in the next notebook.


In [1]:
import json
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

sys.path.append(str(Path("..").resolve()))

from src.config import PROJECT_ROOT, RANDOM_STATE, TARGET_COLUMN, TEST_SIZE
from src.features.feature_selection import get_selected_feature_names, load_feature_selection_decisions

NOTEBOOK_NAME = "13_model_training"
SELECTED_DATA_FILE = PROJECT_ROOT / "data" / "processed" / "creditcard_selected_features.csv"
NOTEBOOK_TABLES_DIR = PROJECT_ROOT / "reports" / "tables" / NOTEBOOK_NAME
NOTEBOOK_FIGURES_DIR = PROJECT_ROOT / "reports" / "figures" / NOTEBOOK_NAME
NOTEBOOK_ARTIFACTS_DIR = PROJECT_ROOT / "artifacts" / NOTEBOOK_NAME

NOTEBOOK_TABLES_DIR.mkdir(parents=True, exist_ok=True)
NOTEBOOK_FIGURES_DIR.mkdir(parents=True, exist_ok=True)
NOTEBOOK_ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

print("Modeling data  :", SELECTED_DATA_FILE)
print("Tables dir     :", NOTEBOOK_TABLES_DIR)
print("Figures dir    :", NOTEBOOK_FIGURES_DIR)
print("Artifacts dir  :", NOTEBOOK_ARTIFACTS_DIR)


Modeling data  : /Users/mohammadmubashir/VCode/Credit-Card-Fraud-Detection/data/processed/creditcard_selected_features.csv
Tables dir     : /Users/mohammadmubashir/VCode/Credit-Card-Fraud-Detection/reports/tables/13_model_training
Figures dir    : /Users/mohammadmubashir/VCode/Credit-Card-Fraud-Detection/reports/figures/13_model_training
Artifacts dir  : /Users/mohammadmubashir/VCode/Credit-Card-Fraud-Detection/artifacts/13_model_training


## 1. Objectives

- Load the finalized selected-feature dataset from notebook `10_feature_selection`.
- Create one reproducible train/test split for the baseline comparison.
- Train baseline `Logistic Regression` and `Random Forest` models.
- Save trained models and test-set probabilities so notebook `14_model_evaluation` can compare them without retraining.

## Output Guide

This notebook writes its main outputs to:

- `reports/tables/13_model_training/`
- `artifacts/13_model_training/`


## 2. Load Final Modeling Data

The baseline models should use only the finalized selected features plus the target column.


In [2]:
df = pd.read_csv(SELECTED_DATA_FILE)
selected_features = get_selected_feature_names(load_feature_selection_decisions())

expected_columns = selected_features + [TARGET_COLUMN]
missing_columns = sorted(set(expected_columns) - set(df.columns))
if missing_columns:
    raise ValueError(f"Selected dataset is missing expected columns: {missing_columns}")

model_df = df.loc[:, expected_columns].copy()
model_df["transaction_id"] = np.arange(len(model_df))

print(f"Selected dataset shape: {model_df.shape}")
print(f"Selected feature count: {len(selected_features)}")
print(f"Fraud rate: {model_df[TARGET_COLUMN].mean():.6f}")
model_df.head()


Selected dataset shape: (283726, 15)
Selected feature count: 13
Fraud rate: 0.001667


,V14_V12_interaction,V14,V17_V16_interaction,V12,V17,V10,V4,V16,V3,V11,V7,V18,log_amount,Class,transaction_id
0,0.192241,-0.311169,-0.097830,-0.617801,0.207971,0.090794,1.378155,-0.470401,2.536347,-0.551600,0.239599,0.025791,5.014760,0,0
1,-0.153151,-0.143772,-0.053260,1.065235,-0.114805,-0.166974,0.448154,0.463917,0.166480,1.612727,-0.078803,-0.183361,1.305626,0,1
2,-0.010966,-0.165946,-3.207904,0.066084,1.109969,0.207643,0.379780,-2.890083,1.773209,0.624501,0.791461,-0.121359,5.939276,0,2
3,-0.051316,-0.287924,0.724897,0.178228,-0.684093,-0.054952,-0.863291,-1.059647,1.792993,-0.226487,0.237609,1.965775,4.824306,0,3
4,-0.602601,-1.119670,0.107008,0.538196,-0.237033,0.753074,0.403034,-0.451449,1.548718,-0.822843,0.592941,-0.038195,4.262539,0,4


## 3. Define Inputs and Target

The notebook keeps the modeling inputs explicit so downstream evaluation can trace exactly which features were used.


In [3]:
X = model_df[selected_features].copy()
y = model_df[TARGET_COLUMN].copy()
row_ids = model_df["transaction_id"].copy()

input_summary = pd.DataFrame(
    [
        {
            "metric": "row_count",
            "value": int(len(model_df)),
        },
        {
            "metric": "feature_count",
            "value": int(len(selected_features)),
        },
        {
            "metric": "target_column",
            "value": TARGET_COLUMN,
        },
        {
            "metric": "fraud_rate",
            "value": float(y.mean()),
        },
    ]
)
input_summary.to_csv(NOTEBOOK_TABLES_DIR / "input_data_summary.csv", index=False)
input_summary


,metric,value
0,row_count,283726
1,feature_count,13
2,target_column,Class
3,fraud_rate,0.001667


## 4. Train/Test Split

The split is stratified because fraud is extremely rare and both baseline models must be compared on the same holdout sample.


In [4]:
X_train, X_test, y_train, y_test, row_id_train, row_id_test = train_test_split(
    X,
    y,
    row_ids,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

split_summary = pd.DataFrame(
    [
        {
            "split": "train",
            "row_count": int(len(X_train)),
            "fraud_count": int(y_train.sum()),
            "fraud_rate": float(y_train.mean()),
        },
        {
            "split": "test",
            "row_count": int(len(X_test)),
            "fraud_count": int(y_test.sum()),
            "fraud_rate": float(y_test.mean()),
        },
    ]
)
split_summary.to_csv(NOTEBOOK_TABLES_DIR / "train_test_split_summary.csv", index=False)
split_summary


,split,row_count,fraud_count,fraud_rate
0,train,226980,378,0.001665
1,test,56746,95,0.001674


## 5. Baseline Preprocessing Setup

- Logistic Regression uses imputation plus scaling because coefficient-based models are sensitive to feature scale.
- Random Forest keeps a lighter preprocessing path because tree-based models do not require scaling.


In [5]:
numeric_features = selected_features.copy()

logreg_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler()),
                ]
            ),
            numeric_features,
        )
    ],
    remainder="drop",
)

rf_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            Pipeline(steps=[("imputer", SimpleImputer(strategy="median"))]),
            numeric_features,
        )
    ],
    remainder="drop",
)

preprocessing_summary = pd.DataFrame(
    [
        {
            "model_name": "logistic_regression",
            "preprocessing": "median_imputation + standard_scaling",
        },
        {
            "model_name": "random_forest",
            "preprocessing": "median_imputation",
        },
    ]
)
preprocessing_summary.to_csv(NOTEBOOK_TABLES_DIR / "preprocessing_summary.csv", index=False)
preprocessing_summary


,model_name,preprocessing
0,logistic_regression,median_imputation + standard_scaling
1,random_forest,median_imputation


## 6. Train Baseline Model 1: Logistic Regression

This is the linear baseline. `class_weight='balanced'` helps the model pay more attention to the rare fraud class during fitting.


In [6]:
logreg_pipeline = Pipeline(
    steps=[
        ("preprocessor", logreg_preprocessor),
        (
            "model",
            LogisticRegression(
                max_iter=2000,
                class_weight="balanced",
                random_state=RANDOM_STATE,
                solver="lbfgs",
            ),
        ),
    ]
)

logreg_pipeline.fit(X_train, y_train)

logreg_train_prob = logreg_pipeline.predict_proba(X_train)[:, 1]
logreg_test_prob = logreg_pipeline.predict_proba(X_test)[:, 1]

print("Logistic Regression training complete.")
pd.Series(logreg_test_prob).describe()


Logistic Regression training complete.


count    5.674600e+04
mean     8.564571e-02
std      1.336449e-01
min      3.193082e-09
25%      2.073688e-02
50%      4.158370e-02
75%      8.621876e-02
max      1.000000e+00
dtype: float64

## 7. Train Baseline Model 2: Random Forest

This is the nonlinear tree baseline. The configuration stays intentionally moderate because the goal here is baseline comparison, not final tuning.


In [7]:
random_forest_pipeline = Pipeline(
    steps=[
        ("preprocessor", rf_preprocessor),
        (
            "model",
            RandomForestClassifier(
                n_estimators=300,
                class_weight="balanced",
                random_state=RANDOM_STATE,
                n_jobs=-1,
            ),
        ),
    ]
)

random_forest_pipeline.fit(X_train, y_train)

rf_train_prob = random_forest_pipeline.predict_proba(X_train)[:, 1]
rf_test_prob = random_forest_pipeline.predict_proba(X_test)[:, 1]

print("Random Forest training complete.")
pd.Series(rf_test_prob).describe()


Random Forest training complete.


count    56746.000000
mean         0.001451
std          0.033146
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max          1.000000
dtype: float64

## 8. Save Training Outputs

Notebook `14_model_evaluation` should not need to retrain these models. This section saves the fitted pipelines, split metadata, and prediction probabilities needed for comparison.


In [8]:
evaluation_predictions = pd.DataFrame(
    {
        "transaction_id": row_id_test.to_numpy(),
        "y_true": y_test.to_numpy(),
        "logistic_regression_probability": logreg_test_prob,
        "random_forest_probability": rf_test_prob,
        "logistic_regression_prediction_0_5": (logreg_test_prob >= 0.5).astype(int),
        "random_forest_prediction_0_5": (rf_test_prob >= 0.5).astype(int),
    }
).sort_values("transaction_id").reset_index(drop=True)

training_predictions = pd.DataFrame(
    {
        "transaction_id": row_id_train.to_numpy(),
        "y_true": y_train.to_numpy(),
        "logistic_regression_probability": logreg_train_prob,
        "random_forest_probability": rf_train_prob,
    }
).sort_values("transaction_id").reset_index(drop=True)

evaluation_predictions.to_csv(NOTEBOOK_TABLES_DIR / "baseline_test_predictions.csv", index=False)
training_predictions.to_csv(NOTEBOOK_TABLES_DIR / "baseline_train_predictions.csv", index=False)

split_indices = {
    "train_transaction_ids": row_id_train.tolist(),
    "test_transaction_ids": row_id_test.tolist(),
}
(NOTEBOOK_ARTIFACTS_DIR / "train_test_split_ids.json").write_text(
    json.dumps(split_indices, indent=2),
    encoding="utf-8",
)

joblib.dump(
    {
        "model": logreg_pipeline,
        "model_name": "logistic_regression",
        "selected_features": selected_features,
        "test_size": TEST_SIZE,
        "random_state": RANDOM_STATE,
    },
    NOTEBOOK_ARTIFACTS_DIR / "baseline_logistic_regression.joblib",
)

joblib.dump(
    {
        "model": random_forest_pipeline,
        "model_name": "random_forest",
        "selected_features": selected_features,
        "test_size": TEST_SIZE,
        "random_state": RANDOM_STATE,
    },
    NOTEBOOK_ARTIFACTS_DIR / "baseline_random_forest.joblib",
)

artifact_manifest = pd.DataFrame(
    [
        {
            "artifact_type": "model",
            "path": str(NOTEBOOK_ARTIFACTS_DIR / "baseline_logistic_regression.joblib"),
        },
        {
            "artifact_type": "model",
            "path": str(NOTEBOOK_ARTIFACTS_DIR / "baseline_random_forest.joblib"),
        },
        {
            "artifact_type": "test_predictions",
            "path": str(NOTEBOOK_TABLES_DIR / "baseline_test_predictions.csv"),
        },
        {
            "artifact_type": "train_predictions",
            "path": str(NOTEBOOK_TABLES_DIR / "baseline_train_predictions.csv"),
        },
        {
            "artifact_type": "split_ids",
            "path": str(NOTEBOOK_ARTIFACTS_DIR / "train_test_split_ids.json"),
        },
    ]
)
artifact_manifest.to_csv(NOTEBOOK_TABLES_DIR / "training_artifact_manifest.csv", index=False)
artifact_manifest


,artifact_type,path
0,model,/Users/mohammadmubashir/VCode/Credit-Card-Frau...
1,model,/Users/mohammadmubashir/VCode/Credit-Card-Frau...
2,test_predictions,/Users/mohammadmubashir/VCode/Credit-Card-Frau...
3,train_predictions,/Users/mohammadmubashir/VCode/Credit-Card-Frau...
4,split_ids,/Users/mohammadmubashir/VCode/Credit-Card-Frau...


## 9. Training Summary

This notebook ends with a compact handoff summary. Detailed comparison belongs in notebook `14_model_evaluation`.


In [9]:
training_overview = pd.DataFrame(
    [
        {
            "model_name": "logistic_regression",
            "status": "trained",
            "uses_class_weight_balanced": True,
            "test_prediction_rows": int(len(evaluation_predictions)),
        },
        {
            "model_name": "random_forest",
            "status": "trained",
            "uses_class_weight_balanced": True,
            "test_prediction_rows": int(len(evaluation_predictions)),
        },
    ]
)
training_overview.to_csv(NOTEBOOK_TABLES_DIR / "training_overview.csv", index=False)

training_report = f"""# Model Training Report

## Purpose
- Train the baseline fraud-detection models on the finalized selected-feature dataset.
- Save the trained artifacts and test-set probabilities for notebook `14_model_evaluation`.

## Training Inputs
- Modeling dataset: `{SELECTED_DATA_FILE}`
- Selected feature count: `{len(selected_features)}`
- Total rows: `{len(model_df)}`
- Fraud rate: `{y.mean():.6f}`
- Test size: `{TEST_SIZE}`
- Random state: `{RANDOM_STATE}`

## Models Trained
- Logistic Regression with median imputation, standard scaling, and `class_weight='balanced'`
- Random Forest with median imputation and `class_weight='balanced'`

## Handoff to Next Notebook
- Evaluate `baseline_test_predictions.csv` in notebook `14_model_evaluation`.
- Compare the two models using precision, recall, F1, PR-AUC, ROC-AUC, and confusion matrices.
"""

report_path = NOTEBOOK_TABLES_DIR / "model_training_report.md"
report_path.write_text(training_report, encoding="utf-8")

print(f"Training report saved to: {report_path}")
training_overview


Training report saved to: /Users/mohammadmubashir/VCode/Credit-Card-Fraud-Detection/reports/tables/13_model_training/model_training_report.md


,model_name,status,uses_class_weight_balanced,test_prediction_rows
0,logistic_regression,trained,True,56746
1,random_forest,trained,True,56746
